

---

# 📘 LeetCode 2010: The Number of Seniors and Juniors to Join the Company II

---

## ❓ Question

A company wants to hire new employees with a salary budget of **$70,000**.  

The hiring criteria are:  
1. Keep hiring the **senior** with the smallest salary until no more seniors can be hired.  
2. Use the remaining budget to hire the **junior** with the smallest salary.  
3. Keep hiring the junior with the smallest salary until no more juniors can be hired.  

Write an SQL query to find the IDs of seniors and juniors hired under the mentioned criteria.  
Return the result table in any order.  

---

## 📊 Sample Data

### Example 1: Candidates Table

| employee_id | experience | salary |
|-------------|------------|--------|
| 1           | Junior     | 10000  |
| 9           | Junior     | 15000  |
| 2           | Senior     | 20000  |
| 11          | Senior     | 16000  |
| 13          | Senior     | 50000  |
| 4           | Junior     | 40000  |

**Output:**

| employee_id |
|-------------|
| 11          |
| 2           |
| 1           |
| 9           |

---

### Example 2: Candidates Table

| employee_id | experience | salary |
|-------------|------------|--------|
| 1           | Junior     | 25000  |
| 9           | Junior     | 10000  |
| 2           | Senior     | 85000  |
| 11          | Senior     | 80000  |
| 13          | Senior     | 90000  |
| 4           | Junior     | 30000  |

**Output:**

| employee_id |
|-------------|
| 9           |
| 1           |
| 4           |

---

## 🏗️ Schema Definition

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

candidates_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("experience", StringType(), False),  # 'Senior' or 'Junior'
    StructField("salary", IntegerType(), False)
])
```

---

## 📥 Data Preparation

```python
# Example 1 data
candidates_data = [
    (1, "Junior", 10000),
    (9, "Junior", 15000),
    (2, "Senior", 20000),
    (11, "Senior", 16000),
    (13, "Senior", 50000),
    (4, "Junior", 40000)
]
```

---

## 🗂️ Create DataFrame

```python
candidates_df = spark.createDataFrame(candidates_data, schema=candidates_schema)
candidates_df.show()
```

---

## 👁️ Register as SQL View

```python
candidates_df.createOrReplaceTempView("Candidates")
```

---



In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

candidates_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("experience", StringType(), False),  # 'Senior' or 'Junior'
    StructField("salary", IntegerType(), False)
])
# Example 1 data
candidates_data = [
    (1, "Junior", 10000),
    (9, "Junior", 15000),
    (2, "Senior", 20000),
    (11, "Senior", 16000),
    (13, "Senior", 50000),
    (4, "Junior", 40000)
]
candidates_df = spark.createDataFrame(candidates_data, schema=candidates_schema)
candidates_df.show()
candidates_df.createOrReplaceTempView("Candidates")


In [0]:
%sql
WITH senior AS (
		SELECT employee_id,
			sum(salary) OVER (
				PARTITION BY experience ORDER BY salary ASC
				) AS cum_sum,
			*
		FROM candidates
		WHERE experience = 'Senior'
		),
	junior AS (
		SELECT employee_id,
			(
				SELECT max(cum_sum)
				FROM senior
				WHERE cum_sum <= 70000
				) + sum(salary) OVER (
				PARTITION BY experience ORDER BY salary ASC
				) AS jun_cum_sal,
			*
		FROM candidates
		WHERE experience = 'Junior'
		)

SELECT employee_id
FROM senior
WHERE cum_sum <= 70000

UNION ALL

SELECT employee_id
FROM junior
WHERE jun_cum_sal <= 70000
